# 04 Stage-2 Oracle ROI Experiment (Phase 0) — detect-then-refine

**Evaluation/experiment notebook. This is Phase 0 of the two-stage detect-then-refine plan
in `docs/small_object_research_notes.md`. NOTHING here is a submission model.**

## What this notebook does

It measures the **upper bound** of a second-stage refiner, using the validation **ground-truth
boxes as a "perfect Stage 1"** (NOT the real YOLO detector):

1. From each GT object, crop a high-resolution ROI from the original image (with context padding).
2. Train a **Stage-2 model** that, given one ROI, predicts **(class, fine mask)** —
   an ImageNet-pretrained lightweight CNN (U-Net + ResNet18 encoder, with a classification head).
3. Place each predicted mask back into full-image coordinates and score **Mask mAP50-95** with the
   **same** evaluator used in `src/03` (mask-IoU -> 10 IoU thresholds 0.5:0.95 -> 101-point AP), at
   **full image resolution**, so the per-class numbers are comparable to the documented V6 run.

## Why an oracle first (the V12/V13 lesson)

Using GT boxes as Stage 1 **decouples two failure modes**: "Stage 1 can't find the small box" vs
"Stage 2 can't refine it". If, *even with perfect boxes*, Stage 2 does not improve the small Caries
classes, the whole pipeline is not worth building. Because Stage 1 is perfect here, every number is
an **upper bound** — the real pipeline (Phase 1) will be lower.

## Pre-registered reading rules (from the research note)

- The **decision metric is the comparable full-image Mask mAP / per-class AP**, NOT ROI-level
  accuracy (the ROI-level numbers in §6 are diagnostic only — same inflation trap as V13 tiled-val).
- Only trust small classes with **adequate support** (Caries 1/2/5, n≈62/73/81). Caries 4 (n=4) and
  Caries 6 (n=5) are noise on their own.
- **Large classes must not regress** — expect Stage-2-on-everything to *hurt* Abrasion/Crown
  (a 224² ROI downsamples a big object), which is exactly why the real system routes large boxes to
  Stage 1. The hybrid upper bound in §7 takes the per-class max to reflect that routing.
- The "how much gain to continue" threshold is intentionally **left open** — read the numbers first.


## 1. Environment Setup

Seed, imports, and `segmentation_models_pytorch` (smp). Training kernels usually have Internet on;
if not, add smp + its weights as an offline wheel/dataset.

In [ ]:
import os, random, math, json
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import yaml

# Stop OpenCV from spawning its own thread pool inside each DataLoader worker — the classic
# cv2 + num_workers>0 deadlock / slowdown. Must be set before any heavy cv2 use.
cv2.setNumThreads(0)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("CWD:", os.getcwd())
print("Kaggle input exists:", Path("/kaggle/input").exists())

In [ ]:
# Install segmentation-models-pytorch if needed (training kernel, Internet usually enabled).
try:
    import segmentation_models_pytorch as smp
    print("smp already installed:", smp.__version__)
except ImportError:
    print("Installing segmentation-models-pytorch ...")
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"])
    import segmentation_models_pytorch as smp
    print("smp installed:", smp.__version__)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__, "| device:", DEVICE)

## 2. Configuration

`ROI_INPUT` is the Stage-2 input size (the lesion is upscaled into this). Padding is the
**context-vs-effective-resolution** ablation knob: bigger padding adds surrounding context but
shrinks the lesion's share of the fixed input. Run the notebook a few times changing only the
padding block to produce the ablation.

In [ ]:
# ---- ROI / padding (the ablation knob) ----
ROI_INPUT      = 224       # Stage-2 input side (lesion upscaled into this)
PAD_MODE       = "relative"  # "relative" (factor x box size) or "absolute" (px margin)
PAD_FACTOR     = 1.5       # used when PAD_MODE == "relative" (try 0.0 tight / 1.5 / 2.0)
PAD_ABS_PX     = 48        # used when PAD_MODE == "absolute" (try 0 / 32 / 64)

# ---- Stage-2 model ----
ENCODER        = "resnet18"
ENCODER_WEIGHTS = "imagenet"   # pretraining matters more than the architecture family
EPOCHS         = 30        # full run; the loop keeps best.pt by val ROI mask-IoU
BATCH_SIZE     = 32
LR             = 1e-3

# ---- DataLoader ----
# ROIs are PRE-CROPPED to /kaggle/working/roi_cache once (each panoramic decoded a single time),
# so each training sample reads a tiny 224-px file instead of decoding a full image. That removes
# both the host-RAM blow-up AND the CPU-bound starvation that left the GPU idle.
NUM_WORKERS    = 4         # tiny-file reads are cheap; lower to 2 if the kernel has few vCPUs

# ---- Eval ----
# Full-resolution mask rasterization to stay comparable to src/03 (slow, like src/03).
IOU_THRESHOLDS = np.linspace(0.5, 0.95, 10)

# ImageNet normalization for the pretrained encoder.
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# V6 per-class Mask mAP50-95 reference, from the src/03 same-code run (see docs).
# Keyed by lowercased class name; looked up against the dataset `names` below.
V6_REF_AP = {
    "abrasion": 0.6471, "crown": 0.6313, "filling": 0.2799,
    "caries 1": 0.1195, "caries 2": 0.0845, "caries 3": 0.0116,
    "caries 4": 0.0000, "caries 5": 0.1097, "caries 6": 0.0051,
}
V6_REF_MAP = 0.2099  # V6 re-scored aggregate (src/03)

def _norm_class_key(nm):
    # Normalize a dataset class name to a V6_REF_AP key, e.g. "Caries 1 class" -> "caries 1".
    # Defined here (not in the eval cell) so BOTH the results table and the save cell can use it
    # regardless of run order.
    s = str(nm).lower().replace("class", "")
    return " ".join(s.split())

print("ROI_INPUT:", ROI_INPUT, "| PAD_MODE:", PAD_MODE,
      "| PAD_FACTOR:", PAD_FACTOR, "| PAD_ABS_PX:", PAD_ABS_PX)
print("Encoder:", ENCODER, ENCODER_WEIGHTS, "| epochs:", EPOCHS, "| batch:", BATCH_SIZE)
print("NUM_WORKERS:", NUM_WORKERS)

## 3. Locate the dataset

Same auto-detection as `src/01`: find `yolo_seg_train.yaml`, read class names, resolve the
train/val image + label folders.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
yaml_candidates = list(INPUT_ROOT.rglob("yolo_seg_train.yaml"))
if not yaml_candidates:
    raise FileNotFoundError("No yolo_seg_train.yaml under /kaggle/input.")
DATA_YAML = yaml_candidates[0]
with open(DATA_YAML, "r", encoding="utf-8") as f:
    dcfg = yaml.safe_load(f)

names = dcfg.get("names")
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
num_classes = len(names)
print("DATA_YAML:", DATA_YAML)
print("num_classes:", num_classes)
print("names:", names)

dataset_root = DATA_YAML.parent
yaml_root = Path(dcfg["path"]) if dcfg.get("path") else dataset_root
if not yaml_root.is_absolute():
    yaml_root = dataset_root / yaml_root

def resolve_split(v):
    if v is None:
        return None
    p = Path(v)
    if p.is_absolute():
        return p
    for cand in (yaml_root / p, dataset_root / p):
        if cand.exists():
            return cand
    return yaml_root / p

train_images = resolve_split(dcfg.get("train"))
val_images   = resolve_split(dcfg.get("val", dcfg.get("valid")))
print("train images:", train_images, train_images.exists())
print("val images  :", val_images, val_images.exists())

def labels_dir_for(images_dir):
    parts = list(Path(images_dir).parts)
    if "images" in parts:
        i = len(parts) - 1 - parts[::-1].index("images")
        parts[i] = "labels"
        return Path(*parts)
    return Path(images_dir).parent / "labels"

## 4. Collect GT objects (the "perfect Stage 1")

Each GT polygon becomes one sample: `(image_path, class_id, polygon[N,2] in normalized coords)`.
The ROI is cropped on the fly in the Dataset (no thousands of files on disk).

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def parse_seg_line(line):
    parts = line.strip().split()
    if len(parts) < 7:
        return None
    try:
        cls = int(float(parts[0]))
        coords = [float(v) for v in parts[1:]]
    except ValueError:
        return None
    if len(coords) % 2:
        coords = coords[:-1]
    poly = np.asarray(coords, dtype=np.float64).reshape(-1, 2)
    return (cls, poly) if len(poly) >= 3 else None

def collect_objects(images_dir):
    lbl_dir = labels_dir_for(images_dir)
    samples = []
    imgs = sorted(p for p in Path(images_dir).iterdir() if p.suffix.lower() in IMG_EXTS)
    for ip in imgs:
        lp = lbl_dir / (ip.stem + ".txt")
        if not lp.exists():
            continue
        for line in lp.read_text().splitlines():
            parsed = parse_seg_line(line)
            if parsed is not None:
                cls, poly = parsed
                samples.append({"image": str(ip), "cls": cls, "poly": poly})
    return samples, imgs

train_samples, train_imgs = collect_objects(train_images)
val_samples, val_imgs = collect_objects(val_images)
print("train objects:", len(train_samples), "over", len(train_imgs), "images")
print("val objects  :", len(val_samples), "over", len(val_imgs), "images")

# Per-class support on val (so we know which classes are trustworthy).
val_support = pd.Series([s["cls"] for s in val_samples]).value_counts().sort_index()
val_support.index = [names[i] for i in val_support.index]
print("\nVal per-class support:")
print(val_support.to_string())

## 5. ROI cropping helpers + on-disk ROI cache

`crop_roi` turns one (image, polygon) into a `ROI_INPUT` square: pad the box (relative or absolute),
make it square around the centre, clamp to the image, crop, resize. It also rasterizes the polygon
into the ROI frame to make the binary GT mask. The same geometry is the mask analog of
`tools/tile_yolo_seg.py::untile_polygon` (crop offset + scale).

The next cell **pre-crops every ROI to `/kaggle/working/roi_cache` once** — grouping samples by
source image so each large panoramic is decoded a *single* time. Training then reads tiny 224-px
files instead of re-decoding full images per sample, which is what frees the GPU (Kaggle-self-contained;
rebuilt fresh each run, no separate Dataset upload).

In [ ]:
def poly_to_pixel_box(poly_norm, w, h):
    xs = poly_norm[:, 0] * w
    ys = poly_norm[:, 1] * h
    return xs.min(), ys.min(), xs.max(), ys.max()

def padded_square_box(x0, y0, x1, y1, w, h):
    bw, bh = x1 - x0, y1 - y0
    cx, cy = (x0 + x1) / 2.0, (y0 + y1) / 2.0
    if PAD_MODE == "relative":
        half = max(bw, bh) * (1.0 + PAD_FACTOR) / 2.0
    else:
        half = max(bw, bh) / 2.0 + PAD_ABS_PX
    half = max(half, 1.0)
    sx0, sy0 = cx - half, cy - half
    sx1, sy1 = cx + half, cy + half
    # Clamp to image while keeping a valid box.
    sx0 = max(0.0, min(sx0, w - 1)); sy0 = max(0.0, min(sy0, h - 1))
    sx1 = max(sx0 + 1, min(sx1, w)); sy1 = max(sy0 + 1, min(sy1, h))
    return int(round(sx0)), int(round(sy0)), int(round(sx1)), int(round(sy1))

def crop_roi(img, poly_norm, with_mask=True):
    h, w = img.shape[:2]
    bx = poly_to_pixel_box(poly_norm, w, h)
    x0, y0, x1, y1 = padded_square_box(*bx, w, h)
    crop = img[y0:y1, x0:x1]
    roi = cv2.resize(crop, (ROI_INPUT, ROI_INPUT), interpolation=cv2.INTER_LINEAR)
    box = (x0, y0, x1, y1)
    if not with_mask:
        return roi, box
    cw, ch = x1 - x0, y1 - y0
    pts = poly_norm.copy()
    pts[:, 0] = (pts[:, 0] * w - x0) / cw * ROI_INPUT
    pts[:, 1] = (pts[:, 1] * h - y0) / ch * ROI_INPUT
    mask = np.zeros((ROI_INPUT, ROI_INPUT), dtype=np.uint8)
    cv2.fillPoly(mask, [pts.astype(np.int32)], 1)
    return roi, box, mask

In [ ]:
import shutil

ROI_CACHE = Path("/kaggle/working/roi_cache")

def build_roi_cache(samples, split):
    # Pre-crop every ROI to disk ONCE. Group by source image so each large panoramic is
    # decoded exactly once (the fix for the CPU-bound, GPU-starved data loading).
    img_dir = ROI_CACHE / split / "img"
    msk_dir = ROI_CACHE / split / "mask"
    img_dir.mkdir(parents=True, exist_ok=True)
    msk_dir.mkdir(parents=True, exist_ok=True)

    by_img = {}
    for idx, s in enumerate(samples):
        by_img.setdefault(s["image"], []).append((idx, s))

    manifest = []
    for image_path, items in by_img.items():
        img = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if img is None:
            print("  WARNING: could not read", image_path)
            continue
        h, w = img.shape[:2]
        for idx, s in items:
            roi, box, mask = crop_roi(img, s["poly"], with_mask=True)
            rf = img_dir / f"{idx:06d}.png"
            mf = msk_dir / f"{idx:06d}.png"
            cv2.imwrite(str(rf), roi)
            cv2.imwrite(str(mf), (mask * 255).astype(np.uint8))
            manifest.append({
                "roi": str(rf), "mask": str(mf), "cls": int(s["cls"]),
                "image": image_path, "box": box, "w": w, "h": h, "poly": s["poly"],
            })
    print(f"  [{split}] cached {len(manifest)} ROIs from {len(by_img)} images -> {ROI_CACHE / split}")
    return manifest

# Rebuild fresh each run (idempotent). ROI crops are tiny, so disk + build time are cheap.
if ROI_CACHE.exists():
    shutil.rmtree(ROI_CACHE)
print("Building ROI cache under", ROI_CACHE, "(each source image decoded exactly once) ...")
train_manifest = build_roi_cache(train_samples, "train")
val_manifest   = build_roi_cache(val_samples, "val")
print("Done. train ROIs:", len(train_manifest), "| val ROIs:", len(val_manifest))

In [ ]:
class ROICacheDataset(Dataset):
    # Reads the tiny pre-cropped ROI files (NOT full panoramic images) -> low CPU, low RAM,
    # so the GPU is no longer starved by per-sample full-image JPEG decoding.
    def __init__(self, manifest, train=False):
        self.m = manifest
        self.train = train
    def __len__(self):
        return len(self.m)
    def __getitem__(self, i):
        e = self.m[i]
        roi = cv2.imread(e["roi"], cv2.IMREAD_COLOR)
        mask = (cv2.imread(e["mask"], cv2.IMREAD_GRAYSCALE) > 127).astype(np.float32)
        roi = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        if self.train and random.random() < 0.5:   # horizontal flip only (light aug)
            roi = roi[:, ::-1, :].copy(); mask = mask[:, ::-1].copy()
        roi = (roi - IMAGENET_MEAN) / IMAGENET_STD
        roi = torch.from_numpy(roi.transpose(2, 0, 1))
        mask = torch.from_numpy(mask)[None]
        return roi, mask, e["cls"]

train_ds = ROICacheDataset(train_manifest, train=True)
val_ds   = ROICacheDataset(val_manifest, train=False)

loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
if NUM_WORKERS > 0:
    loader_kw.update(persistent_workers=True, prefetch_factor=4)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **loader_kw)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kw)
print("train ROIs:", len(train_ds), "| val ROIs:", len(val_ds), "| workers:", NUM_WORKERS)

In [ ]:
import segmentation_models_pytorch as smp

# U-Net (binary lesion mask) + classification head (aux_params) -> multi-task class + fine seg.
model = smp.Unet(
    encoder_name=ENCODER,
    encoder_weights=ENCODER_WEIGHTS,
    in_channels=3,
    classes=1,
    activation=None,
    aux_params=dict(classes=num_classes, dropout=0.2),
).to(DEVICE)

dice_loss = smp.losses.DiceLoss(mode="binary", from_logits=True)
bce_loss  = nn.BCEWithLogitsLoss()
ce_loss   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

def seg_cls_loss(mask_logits, cls_logits, mask_gt, cls_gt):
    lseg = dice_loss(mask_logits, mask_gt) + bce_loss(mask_logits, mask_gt)
    lcls = ce_loss(cls_logits, cls_gt)
    return lseg + lcls, lseg.item(), lcls.item()

print(model.__class__.__name__, "built |", ENCODER, ENCODER_WEIGHTS)

## 6. Train Stage-2 + ROI-level diagnostics

The per-ROI accuracy / mask-IoU printed here are **diagnostic only** (ROI-level = inflated). The
real decision metric is the comparable full-image Mask mAP in §7.

In [ ]:
@torch.no_grad()
def roi_level_eval():
    model.eval()
    correct = total = 0
    iou_sum = 0.0
    per_cls_correct = np.zeros(num_classes); per_cls_total = np.zeros(num_classes)
    for roi, mask, cls in val_loader:
        roi = roi.to(DEVICE); mask = mask.to(DEVICE)
        cls_t = torch.as_tensor(cls).to(DEVICE)
        mlog, clog = model(roi)
        pred_cls = clog.argmax(1)
        correct += (pred_cls == cls_t).sum().item(); total += len(cls_t)
        for c, ok in zip(cls.tolist(), (pred_cls == cls_t).tolist()):
            per_cls_total[c] += 1; per_cls_correct[c] += int(ok)
        pm = (torch.sigmoid(mlog) > 0.5).float()
        inter = (pm * mask).sum((1, 2, 3))
        union = ((pm + mask) > 0).float().sum((1, 2, 3)).clamp(min=1)
        iou_sum += (inter / union).sum().item()
    return correct / max(total, 1), iou_sum / max(total, 1), per_cls_correct, per_cls_total

best_iou = -1.0
n_batches = len(train_loader)
history = []   # per-epoch curve, saved to disk in §9 for background-run analysis
for epoch in range(1, EPOCHS + 1):
    print(f"\n========== Epoch {epoch}/{EPOCHS} started ({n_batches} batches) ==========", flush=True)
    model.train()
    run = 0.0
    for roi, mask, cls in train_loader:
        roi = roi.to(DEVICE); mask = mask.to(DEVICE)
        cls_t = torch.as_tensor(cls).to(DEVICE)
        optimizer.zero_grad()
        mlog, clog = model(roi)
        loss, ls, lc = seg_cls_loss(mlog, clog, mask, cls_t)
        loss.backward(); optimizer.step()
        run += loss.item()
    train_loss = run / n_batches
    acc, miou, pcc, pct = roi_level_eval()
    history.append({"epoch": epoch, "train_loss": train_loss,
                    "val_roi_acc": acc, "val_roi_mask_iou": miou})
    print(f"Epoch {epoch}/{EPOCHS} finished | train_loss {train_loss:.4f} | "
          f"val ROI-acc {acc:.3f} | val ROI mask-IoU {miou:.3f}")
    if miou > best_iou:
        best_iou = miou
        torch.save(model.state_dict(), "/kaggle/working/stage2_best.pt")
        print(f"  -> new best ROI mask-IoU {miou:.3f}; saved stage2_best.pt")

# Persist the training curve (epoch, loss, val acc/IoU) for later analysis.
import pandas as _pd
_pd.DataFrame(history).to_csv("/kaggle/working/stage2_history.csv", index=False)
print("Saved training history -> /kaggle/working/stage2_history.csv")

model.load_state_dict(torch.load("/kaggle/working/stage2_best.pt"))
acc, miou, pcc, pct = roi_level_eval()
print("\n[ROI-level diagnostic — NOT the decision metric] per-class classification acc:")
for c in range(num_classes):
    if pct[c] > 0:
        print(f"  {names[c]:>10s} (n={int(pct[c])}): acc {pcc[c]/pct[c]:.3f}")

## 7. Comparable full-image Mask mAP (the decision metric)

Place every Stage-2 ROI mask back into a full-image canvas (crop offset + resize = the mask analog
of `untile_polygon`), then score with the same mask-mAP code as `src/03` at full resolution.
Each GT box yields exactly one prediction (perfect recall by construction) — so this is an **upper
bound**. Per-class AP is compared to the documented V6 (src/03, same evaluator).

In [ ]:
@torch.no_grad()
def stage2_predict_batch(entries):
    # Batch ALL ROIs of one image through the model in a single forward (opt #3).
    # Returns per-object (pred_cls, conf, pred_box, pred_mask_local) where pred_mask_local is the
    # binary mask at the box's own pixel size — NO full-image canvas is ever allocated (opt #2).
    xs = []
    for e in entries:
        roi = cv2.imread(e["roi"], cv2.IMREAD_COLOR)
        roi = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        roi = (roi - IMAGENET_MEAN) / IMAGENET_STD
        xs.append(roi.transpose(2, 0, 1))
    batch = torch.from_numpy(np.stack(xs)).to(DEVICE)
    mlog, clog = model(batch)
    prob = torch.softmax(clog, 1)
    masks = (torch.sigmoid(mlog)[:, 0] > 0.5).cpu().numpy().astype(np.uint8)
    out = []
    for i, e in enumerate(entries):
        pc = int(prob[i].argmax().item()); cf = float(prob[i].max().item())
        x0, y0, x1, y1 = e["box"]
        pm = cv2.resize(masks[i], (x1 - x0, y1 - y0), interpolation=cv2.INTER_NEAREST)
        out.append((pc, cf, (x0, y0, x1, y1), pm))
    return out

def gt_local_mask(poly_norm, w, h):
    # Rasterize a GT polygon in its OWN bbox-local frame (no full-image array).
    pts = poly_norm.copy()
    pts[:, 0] *= w; pts[:, 1] *= h
    gx0 = max(0, int(np.floor(pts[:, 0].min()))); gy0 = max(0, int(np.floor(pts[:, 1].min())))
    gx1 = min(w, int(np.ceil(pts[:, 0].max())));  gy1 = min(h, int(np.ceil(pts[:, 1].max())))
    gw = max(1, gx1 - gx0); gh = max(1, gy1 - gy0)
    m = np.zeros((gh, gw), dtype=np.uint8)
    pts[:, 0] -= gx0; pts[:, 1] -= gy0
    cv2.fillPoly(m, [pts.astype(np.int32)], 1)
    return (gx0, gy0, gx1, gy1), m

def iou_local(pbox, pmask, gbox, gmask):
    # IoU from mask areas + an intersection computed only over the overlap rectangle (opt #2).
    pa = int(pmask.sum()); ga = int(gmask.sum())
    if pa == 0 or ga == 0:
        return 0.0
    ix0 = max(pbox[0], gbox[0]); iy0 = max(pbox[1], gbox[1])
    ix1 = min(pbox[2], gbox[2]); iy1 = min(pbox[3], gbox[3])
    if ix1 <= ix0 or iy1 <= iy0:
        inter = 0
    else:
        pc = pmask[iy0 - pbox[1]:iy1 - pbox[1], ix0 - pbox[0]:ix1 - pbox[0]]
        gc = gmask[iy0 - gbox[1]:iy1 - gbox[1], ix0 - gbox[0]:ix1 - gbox[0]]
        inter = int(np.logical_and(pc, gc).sum())
    union = pa + ga - inter
    return inter / union if union > 0 else 0.0

# Group cached val entries by source image for per-image matching.
by_image = {}
for e in val_manifest:
    by_image.setdefault(e["image"], []).append(e)

records = {(c, ti): [] for c in range(num_classes) for ti in range(len(IOU_THRESHOLDS))}
n_gt = np.zeros(num_classes, dtype=int)

model.eval()
for image_path, entries in by_image.items():
    w, h = entries[0]["w"], entries[0]["h"]
    preds = stage2_predict_batch(entries)                       # one forward per image (opt #3)
    gts = [(e["cls"], *gt_local_mask(e["poly"], w, h)) for e in entries]
    for gc, _, _ in gts:
        n_gt[gc] += 1
    # Cache the IoU matrix ONCE; the 10 thresholds only do comparisons on it (opt #1).
    iou_mat = np.zeros((len(preds), len(gts)), dtype=np.float32)
    for pi, (pc, cf, pbox, pm) in enumerate(preds):
        for gi, (gc, gbox, gm) in enumerate(gts):
            if pc == gc:
                iou_mat[pi, gi] = iou_local(pbox, pm, gbox, gm)
    for ti, thr in enumerate(IOU_THRESHOLDS):
        order = sorted(range(len(preds)), key=lambda i: preds[i][1], reverse=True)
        used = [False] * len(gts)
        for pi in order:
            pc, cf = preds[pi][0], preds[pi][1]
            best_iou_v, best_g = 0.0, -1
            for gi in range(len(gts)):
                if used[gi] or gts[gi][0] != pc:
                    continue
                v = iou_mat[pi, gi]
                if v > best_iou_v:
                    best_iou_v, best_g = v, gi
            tp = 1 if (best_g >= 0 and best_iou_v >= thr) else 0
            if tp:
                used[best_g] = True
            records[(pc, ti)].append((cf, tp))
print("Accumulated match records over", len(by_image), "val images.")

In [ ]:
def ap_101(confs, tps, npos):
    if npos == 0:
        return float("nan")
    if len(confs) == 0:
        return 0.0
    o = np.argsort(-np.asarray(confs))
    tps = np.asarray(tps)[o]
    tp_cum = np.cumsum(tps); fp_cum = np.cumsum(1 - tps)
    rec = tp_cum / npos
    prec = tp_cum / np.maximum(tp_cum + fp_cum, 1e-9)
    ap = 0.0
    for r in np.linspace(0, 1, 101):
        p = prec[rec >= r].max() if np.any(rec >= r) else 0.0
        ap += p / 101.0
    return ap

per_class_ap = np.full(num_classes, np.nan)
for c in range(num_classes):
    if n_gt[c] == 0:
        continue
    aps = []
    for ti in range(len(IOU_THRESHOLDS)):
        recs = records[(c, ti)]
        confs = [r[0] for r in recs]; tps = [r[1] for r in recs]
        aps.append(ap_101(confs, tps, n_gt[c]))
    per_class_ap[c] = np.nanmean(aps)

valid = ~np.isnan(per_class_ap)
oracle_map = float(np.nanmean(per_class_ap[valid]))

rows = []
for c in range(num_classes):
    nm = names[c]
    v6 = V6_REF_AP.get(_norm_class_key(nm), np.nan)
    rows.append({
        "class": nm, "n_gt": int(n_gt[c]),
        "stage2_oracle_AP": round(per_class_ap[c], 4) if not np.isnan(per_class_ap[c]) else np.nan,
        "V6_ref_AP(src/03)": v6,
        "delta": round(per_class_ap[c] - v6, 4) if (not np.isnan(per_class_ap[c]) and not np.isnan(v6)) else np.nan,
    })
table = pd.DataFrame(rows)
print(table.to_string(index=False))
print(f"\nStage-2 oracle Mask mAP50-95 (upper bound): {oracle_map:.4f}")
print(f"V6 re-scored Mask mAP50-95 (src/03 reference): {V6_REF_MAP:.4f}")

# Hybrid upper bound: route large classes to V6, small classes to Stage-2 (per-class max-by-routing).
hybrid = []
for c in range(num_classes):
    v6 = V6_REF_AP.get(_norm_class_key(names[c]), np.nan)
    s2 = per_class_ap[c]
    vals = [v for v in (v6, s2) if not np.isnan(v)]
    if vals:
        hybrid.append(max(vals))  # routing keeps whichever path is better for that class
print(f"Hybrid (route large->V6, small->Stage2) upper-bound mAP: {np.mean(hybrid):.4f}")

## 8. How to read this

- **Oracle Mask mAP is an UPPER BOUND** (perfect Stage 1). The real pipeline will be lower once a
  real detector's small-box recall is factored in — set any "continue" bar above that haircut.
- **Look at the small classes with support** (Caries 1/2/5). If their `delta` vs V6 is clearly
  positive and beyond the ~0.003 noise band, high-res ROI refinement is worth pursuing.
- **Expect large classes (Abrasion/Crown) to be NEGATIVE** here — a 224² ROI downsamples them; this
  validates routing them to Stage 1. The **hybrid** number reflects that routing.
- Re-run §2 with `PAD_FACTOR` 0.0 / 1.5 / 2.0 (and/or `PAD_MODE="absolute"`) to get the
  context-vs-resolution ablation, keeping everything else fixed.
- Decision threshold is intentionally open — discuss the numbers before building Phase 1.


## 9. Save outputs for later analysis (Kaggle "Save Version")

Persists everything useful to `/kaggle/working` so a background commit run leaves downloadable
output files:

- `stage2_best.pt` — best Stage-2 checkpoint (by val ROI mask-IoU);
- `stage2_history.csv` — per-epoch train loss + val ROI acc/IoU (overfitting check);
- `stage2_results.csv` — the per-class decision-metric table (oracle AP vs V6, delta);
- `stage2_summary.json` — self-describing run record: **config (padding etc.) + aggregate metrics +
  per-class numbers**, so each Save Version is traceable when you sweep the padding ablation.

In [ ]:
# Persist all useful artifacts to /kaggle/working so a background "Save Version" run leaves
# self-describing output files for later analysis.

# 1) per-class decision-metric table
RESULTS_CSV = "/kaggle/working/stage2_results.csv"
table.to_csv(RESULTS_CSV, index=False)

# 2) hybrid upper bound (route large->V6, small->Stage2)
hybrid_vals = []
for c in range(num_classes):
    v6 = V6_REF_AP.get(_norm_class_key(names[c]), np.nan)
    s2 = per_class_ap[c]
    vv = [v for v in (v6, s2) if not np.isnan(v)]
    if vv:
        hybrid_vals.append(max(vv))
hybrid_map = float(np.mean(hybrid_vals)) if hybrid_vals else float("nan")

# 3) self-describing run summary (config + metrics + per-class), so each Kaggle version is traceable
def _f(x):
    return None if (x is None or (isinstance(x, float) and np.isnan(x))) else float(x)

summary = {
    "config": {
        "ROI_INPUT": ROI_INPUT, "PAD_MODE": PAD_MODE, "PAD_FACTOR": PAD_FACTOR,
        "PAD_ABS_PX": PAD_ABS_PX, "ENCODER": ENCODER, "ENCODER_WEIGHTS": ENCODER_WEIGHTS,
        "EPOCHS": EPOCHS, "BATCH_SIZE": BATCH_SIZE, "LR": LR, "SEED": SEED,
    },
    "metrics": {
        "stage2_oracle_mAP50_95": _f(oracle_map),
        "v6_rescored_mAP50_95": _f(V6_REF_MAP),
        "hybrid_upper_bound_mAP50_95": _f(hybrid_map),
        "best_val_roi_mask_iou": _f(best_iou),
    },
    "per_class": [
        {
            "class": names[c],
            "n_gt": int(n_gt[c]),
            "stage2_oracle_AP": _f(per_class_ap[c]),
            "v6_ref_AP": _f(V6_REF_AP.get(_norm_class_key(names[c]), np.nan)),
            "roi_level_acc": _f(pcc[c] / pct[c]) if pct[c] > 0 else None,
        }
        for c in range(num_classes)
    ],
}
SUMMARY_JSON = "/kaggle/working/stage2_summary.json"
with open(SUMMARY_JSON, "w") as f:
    json.dump(summary, f, indent=2)

# 4) report what landed in the output directory
print("Outputs written to /kaggle/working:")
for p in ["stage2_best.pt", "stage2_history.csv", "stage2_results.csv", "stage2_summary.json"]:
    fp = Path("/kaggle/working") / p
    status = f"OK   ({fp.stat().st_size} bytes)" if fp.exists() else "MISSING"
    print(f"  [{status}] {p}")
print("\nMetrics:")
print(json.dumps(summary["metrics"], indent=2))